### This script is used to identify outliers in raw data datasets:
### assuming log-normally distributed data
### each value further than 6*standard deviation from the mean on log-scale is considered as outlier:



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import plotnine as p9
import os

#### create output folders:

In [2]:
# Define the output  folder paths:
folder_path = '../../output_data/outlier_removed_raw_data'
folder_path1 = '../../output_data/outlier_removed_raw_data/outlier_removed'
folder_path2 = '../../output_data/outlier_removed_raw_data/outlier_flagged'

# Check if the folder exists, if not, create it:
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

if not os.path.exists(folder_path1):
    os.makedirs(folder_path1)

if not os.path.exists(folder_path2):
    os.makedirs(folder_path2)

In [3]:
#--- List files ---
inputFolder = '../../input_data/raw_quality/'
files = glob(f'{inputFolder}*csv')
print([file.split('_')[-1].split('.')[0] for file in files])


['arcticdeltas', 'denmark', 'france', 'germany', 'GRQA', 'sweden', 'MU']


In [4]:
# create empty lists for loop: 
list_out = []
list_long = []
list_outliers_data = []
list_outliers_removed = []
table_df = pd.DataFrame(columns=['dataset', 'fraction', 'n_obs', 'median', 'min', 'max', 'low_thres', 'up_thres', 'n_outliers', 'up_out', 'low_out', 'percent','min_without_outl', 'max_without_outl'])
outliers_table = pd.DataFrame()

In [5]:
# --- Analyse raw data ---detect outliers

columns_fractions = ['NO3N', 'NO2N', 'NO2N_NO3N', 'NH4N', 'DIN', 'TOC', 'DOC', 'TP', 'DIP', 'OPO4']

first_iteration = True
for file in files:
    dataset_name = file.split('_')[-1].split('.')[0]
    delimiter = ','
    decimal = '.'

    # if dataset_name is denmark: delimiter = ';' and decimal = ',':
    if dataset_name in['denmark']:
        delimiter = ';'
    if dataset_name=='denmark':
        decimal = ','
    raw_data = pd.read_csv(file,delimiter=delimiter,decimal=decimal)

    #--- Column name consistency ---
    if dataset_name=='MU':
        raw_data['country'] = 'USA'
        dataset_name = 'USGS'
        raw_data = raw_data.rename(columns = {'NH4':'NH4N'})
        raw_data = raw_data[raw_data.columns.drop('country')]
    to_replace = {'SiteID':'site_id','Station':'site_id','date_string':'obs_date','Date':'obs_date'}
    raw_data = raw_data.rename(columns=to_replace)

   
    #--- Print some information ---
    print('\n',dataset_name,raw_data.columns.tolist())
    raw_data_2 = raw_data.copy()
    raw_data_2 = raw_data_2.copy().replace(-9999,np.nan)
   
    raw_data_2['obs_date'] = pd.to_datetime(raw_data_2['obs_date'])
    raw_data_2['year'] = raw_data_2['obs_date'].dt.year

    if 'X' in raw_data_2.columns:
        raw_data_2 = raw_data_2.drop(columns = ['X'])
    if 'Unnamed: 0' in raw_data_2.columns:
        raw_data_2 = raw_data_2.drop(columns = ['Unnamed: 0'])
    #if dataset_name == 'germany':
     #   raw_data_2 = raw_data_2.drop(columns = ['NO3N_F', 'NH4N_F', 'NO2N_F', 'TOC_F', 'DOC_F', 'TP_F', 'DIP_F'])

    if dataset_name == 'arcticdeltas':
        raw_data_2['NO3N'] =pd.to_numeric(raw_data_2['NO3N'], errors = 'coerce').astype(float)
        raw_data_2['NH4N'] =pd.to_numeric(raw_data_2['NH4N'], errors = 'coerce').astype(float)

    if dataset_name == 'france':
        not_consistent_DOC = [2094900,2099500,2074000, 2060750,2057000,2114000,2047000,2036000, 5222000, 2106600, 2089900, 2070250, 5046000, 2066000, 2042000, 2044000, 2071050, 2038000, 2112000, 2058000, 2043000]
        condition = raw_data_2['site_id'].isin(not_consistent_DOC)
        raw_data_2.loc[condition, ['NO3', 'DIP', 'DOC']] = pd.NA


    # print stats per compound and dataset:
    print(raw_data_2.replace(-9999,np.nan).describe())
    
   
    #--- Append datasets ---
    raw_data_2['dataset'] = dataset_name
    list_out.append(raw_data.copy())

    # create copy of raw_data_2, where negative values, zero values and outliers are replaced by nans instead of being flagged:
    raw_data_outliers_rm = raw_data_2.copy()

    columns_ignore =['obs_date', 'year', 'site_id', 'dataset', 'NO3N_F', 'NH4N_F', 'NO2N_F', 'TOC_F', 'DOC_F', 'TP_F', 'DIP_F', 'Q']
    for column in raw_data_2.columns:
        if column not in columns_ignore and raw_data_2[column].count() > 0:
            column_fraction = raw_data_2[column]

            # select only values > zero and remove NA values:
            column_fraction_1 =column_fraction[column_fraction>0].dropna()

            log_mean = np.log(column_fraction_1).mean()
            log_sd = np.log(column_fraction_1).std()

            max_value = log_mean + (6*log_sd)
            min_value = log_mean - (6*log_sd)
            up_thres = np.exp(max_value)
            low_thres = np.exp(min_value)
            #plt.boxplot(np.log(column_fraction_1), notch=None, sym=None, vert=None, whis=True, positions=None, widths=None, patch_artist=None, 
            #                 bootstrap=None, usermedians=None, conf_intervals=None, meanline=None, showmeans=True, showcaps=None, 
            #                 showbox=None, showfliers=True, boxprops=None, labels=None, flierprops=None, medianprops=None, meanprops=None, 
            #                 capprops=None, whiskerprops=None, manage_ticks=True, autorange=False, zorder=None, data=None)
            
            #plt.plot(1, np.log(column_fraction_1).mean()+6*np.log(column_fraction_1).std(),'ro')
            #plt.plot(1, np.log(column_fraction_1).mean()-6*np.log(column_fraction_1).std(),'ro')
            #plt.title(f"Boxplot log-scale {dataset_name} for {column}")
            #plt.show()
            
    

            # and additional a hitogram plot:
            #plt.hist(column_fraction_1, bins = 100)
            #plt.axvline(low_thres, color='r', linestyle='dashed', linewidth=1)
            #plt.axvline(up_thres, color='r', linestyle='dashed', linewidth=1)
            #plt.axvline(column_fraction_1.max(), color='k', linestyle='dashed', linewidth=1)
            #plt.axvline(column_fraction_1.min(), color='k', linestyle='dashed', linewidth=1)
            low_thres_str = '{:.3f}'.format(low_thres)  
            up_thres_str = '{:.3f}'.format(up_thres)    

            # Positionen für den Text
            #y_pos_low = plt.ylim()[1] * 0.9  
            #y_pos_up = plt.ylim()[1] * 0.8  
            #y_pos_min = plt.ylim()[1] * 0.7 
            #y_pos_max = plt.ylim()[1] * 0.6
            #y_pos_median = plt.ylim()[1] * 0.5
            up_out = len(column_fraction_1[column_fraction_1>up_thres])
            low_out = len(column_fraction_1[column_fraction_1<low_thres])
            n_outliers = up_out +low_out
            percent = "{:.3f}".format(n_outliers/len(column_fraction_1))



            # after removal of outliers min and max:
            max_after_rm = column_fraction_1[column_fraction_1<=up_thres].max()
            min_after_rm = column_fraction_1[column_fraction_1>=low_thres].min()
            
            # create a table: dataset, fraction, n_obs, median, min, max, low_thres, up_thres, n_ouliers, up_out, low_out, percent
            row_values = {'dataset': dataset_name, 'fraction': column, 'n_obs': len(column_fraction_1), 'median':column_fraction_1.median(), 'min': column_fraction_1.min(),
                                'max': column_fraction_1.max(), 'low_thres': low_thres_str, 'up_thres': up_thres_str, 'n_outliers': n_outliers,
                                'up_out': up_out, 'low_out': low_out, 'percent': percent, 'min_without_outl':min_after_rm, 'max_without_outl':max_after_rm}
            table_df = pd.concat([table_df, pd.DataFrame([row_values])], ignore_index=True)


            
            

            # - values larger mean + 6 standard deviations and mean - 6 standard deviations are considered as outliers: on log scale!!!!
            log_mean = np.log(column_fraction_1).mean()
            log_sd = np.log(column_fraction_1).std()

            max_value = log_mean + (6*log_sd)
            min_value = log_mean - (6*log_sd)

            # Outliers are flagged: if smaller than min_value on log scale by 1, if larger than max_value on log scale by 2, if values are zero by 3 and if negative obs_values by -1:
            raw_data_2[f"{column}_F_log_6sd"] = np.nan
            raw_data_2[f"{column}_F_log_6sd"] = np.where(raw_data_2[column]==0, 3,
                                                np.where(raw_data_2[column]<0, -1,
                                                np.where(np.log(raw_data_2[column])<min_value, 1, 
                                                np.where(np.log(raw_data_2[column])>max_value,2,
                                                0))))

    
           
            
            # in dataframe raw_data_outliers_rm outliers and zero and negative values are removed:
            raw_data_outliers_rm.loc[(np.log(raw_data_outliers_rm[column]) < min_value) | (np.log(raw_data_outliers_rm[column] )> max_value) | (raw_data_outliers_rm[column] <= 0) , column] = np.nan
            
    # append table_df to dataframe: outliers table
    outliers_table = pd.concat([outliers_table, table_df])
    table_df = pd.DataFrame(columns=['dataset', 'fraction', 'n_obs', 'median', 'min', 'max', 'low_thres', 'up_thres', 'n_outliers', 'up_out', 'low_out', 'percent'])
    
    # create copy of dataframe as raw_data_outliers_flagged
    raw_data_outliers_flagged = raw_data_2.copy()
   
    list_outliers_data.append(raw_data_outliers_flagged.copy())

    # Now drop all columns which have in all columns of interest only nans after removing outliers:
    columns_subset = [col for col in columns_fractions if col in raw_data_outliers_rm.columns]
    # Drop rows where all columns in the subset are NaNs
    raw_data_outliers_rm.dropna(subset=columns_subset, how='all', inplace=True)
    list_outliers_removed.append(raw_data_outliers_rm.copy())       






    # save files as csv: 
    raw_data_outliers_rm.to_csv(f'../../output_data/outlier_removed_raw_data/outlier_removed/outlier_removed_raw_data_{dataset_name}.csv')    
    raw_data_outliers_flagged.to_csv(f'../../output_data/outlier_removed_raw_data/outlier_flagged/outlier_flagged_raw_data_{dataset_name}.csv')       



 arcticdeltas ['Unnamed: 0', 'site_id', 'obs_date', 'NO3N', 'NH4N', 'NO2N', 'TOC', 'DOC', 'TP', 'DIP']
                            obs_date        NO3N        NH4N  NO2N  TOC  \
count                            521  518.000000  520.000000   0.0  0.0   
mean   2013-08-13 15:42:29.712091904    0.093032    0.032637   NaN  NaN   
min              2003-06-18 00:00:00   -0.000467    0.000000   NaN  NaN   
25%              2009-12-01 00:00:00    0.032000    0.005206   NaN  NaN   
50%              2014-04-26 00:00:00    0.070921    0.010000   NaN  NaN   
75%              2017-12-14 00:00:00    0.136058    0.024000   NaN  NaN   
max              2021-12-14 00:00:00    0.941242    0.669520   NaN  NaN   
std                              NaN    0.089527    0.074840   NaN  NaN   

              DOC   TP         DIP         year  
count  521.000000  0.0  419.000000   521.000000  
mean     0.006701  NaN    0.007111  2013.101727  
min      0.000000  NaN    0.000000  2003.000000  
25%      0.004000  N

C:\Users\bartusch\AppData\Local\Temp\ipykernel_2688\3175264362.py:135: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encou


 denmark ['Unnamed: 0', 'site_id', 'obs_date', 'NO3N', 'NH4N', 'NO2N', 'TOC', 'DOC', 'TP', 'DIP']
            site_id                       obs_date           NO3N  \
count  2.959400e+05                         295940  220269.000000   
mean   3.252827e+07  2001-07-24 06:54:25.764682368       3.966326   
min    1.000039e+06            1970-01-05 00:00:00       0.000000   
25%    2.100068e+07            1993-04-21 00:00:00       1.700000   
50%    3.200002e+07            2000-07-18 00:00:00       3.270000   
75%    4.500004e+07            2010-03-30 00:00:00       5.510000   
max    5.600000e+07            2022-04-29 00:00:00      65.000000   
std    1.402276e+07                            NaN       3.201782   

                NH4N  NO2N          TOC          DOC             TP  \
count  208996.000000   0.0  1369.000000  2455.000000  285771.000000   
mean        0.233831   NaN     9.368503     6.644312       0.216579   
min         0.000000   NaN     1.300000     0.100000       0.00000

C:\Users\bartusch\AppData\Local\Temp\ipykernel_2688\3175264362.py:135: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
c:\Users\bartusch\cnp_env_local\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log



 france ['X', 'site_id', 'obs_date', 'NO3N', 'NH4N', 'NO2N', 'TOC', 'DOC', 'TP', 'DIP']
            site_id                       obs_date           NO3N  NH4N  NO2N  \
count  1.624720e+05                         162472  152857.000000   0.0   0.0   
mean   3.970722e+06  1999-10-27 01:52:31.528878848       3.170660   NaN   NaN   
min    1.001128e+06            1969-04-02 00:00:00       0.000000   NaN   NaN   
25%    3.113040e+06            1993-06-14 00:00:00       1.174610   NaN   NaN   
50%    4.090000e+06            2001-04-17 00:00:00       2.394398   NaN   NaN   
75%    5.093550e+06            2008-03-19 00:00:00       4.517732   NaN   NaN   
max    6.213000e+06            2016-06-29 00:00:00      37.948950   NaN   NaN   
std    1.308356e+06                            NaN       2.597324   NaN   NaN   

       TOC            DOC   TP            DIP           year  
count  0.0   96395.000000  0.0  144256.000000  162472.000000  
mean   NaN      64.979313  NaN       0.116211    1999.3

...

C:\Users\bartusch\AppData\Local\Temp\ipykernel_2688\3175264362.py:135: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



 sweden ['site_id', 'obs_date', 'NO2N_NO3N', 'NH4N', 'DIN', 'TOC', 'DOC', 'TP', 'DIP']
            site_id                       obs_date    NO2N_NO3N         NH4N  \
count  11096.000000                          11096  5412.000000  5815.000000   
mean      14.353280  2010-01-23 15:07:00.475847168     0.025277     0.023420   
min        1.000000            1985-07-11 00:00:00     0.000885     0.000040   
25%        4.000000            2003-06-24 00:00:00     0.010570     0.007650   
50%        7.000000            2010-11-27 12:00:00     0.017250     0.013007   
75%       15.000000            2018-04-27 00:00:00     0.029920     0.024405   
max       66.000000            2022-08-16 00:00:00     0.787440     0.434390   
std       17.056038                            NaN     0.030190     0.036631   

              DIN          TOC          DOC          TP          DIP  \
count  100.000000  4686.000000  5660.000000  144.000000  5309.000000   
mean     0.024530    19.591234    21.575758    

C:\Users\bartusch\AppData\Local\Temp\ipykernel_2688\3175264362.py:135: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



 USGS ['Unnamed: 0', 'site_id', 'obs_date', 'Q', 'NO3N', 'TOC', 'OPO4', 'NH4N']
                            obs_date              Q           NO3N  \
count                         284270  284270.000000  232435.000000   
mean   1997-11-13 19:41:24.654729472     270.905090       2.184638   
min              1965-10-02 00:00:00       0.000000       0.001000   
25%              1990-01-21 00:00:00       0.849505       0.383000   
50%              2000-02-03 00:00:00       9.967530       1.154000   
75%              2006-10-11 00:00:00      62.013894       3.020000   
max              2013-09-30 00:00:00   63712.904330      71.428000   
std                              NaN    1682.086390       2.657857   

                TOC           OPO4           NH4N           year  
count  43516.000000  139759.000000  105863.000000  284270.000000  
mean       6.277028       0.082734       0.150533    1997.377043  
min        0.010000       0.000500       0.000003    1965.000000  
25%        2.700000 

C:\Users\bartusch\AppData\Local\Temp\ipykernel_2688\3175264362.py:135: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


#### Table summarizing the number of outliers:

In [6]:
outliers_table.sort_values(by = 'fraction')

,dataset,fraction,n_obs,median,min,max,low_thres,up_thres,n_outliers,up_out,low_out,percent,min_without_outl,max_without_outl
2,sweden,DIN,100,0.016000,1.000000e-03,0.236000,0.000,13.676,0,0,0,0.000,0.001000,0.236000
3,arcticdeltas,DIP,413,0.004850,2.898800e-04,0.042000,0.000,0.869,0,0,0,0.000,0.000290,0.042000
6,germany,DIP,257656,0.050000,9.000000e-07,80.000000,0.000,21.893,44,2,42,0.000,0.000300,17.000000
6,GRQA,DIP,605415,0.024995,1.548650e-04,404.039130,0.000,500.663,0,0,0,0.000,0.000155,404.039130
6,sweden,DIP,5309,0.003710,1.000000e-05,0.729900,0.000,1.384,1,0,1,0.000,0.000020,0.729900
5,denmark,DIP,256626,0.047000,3.400000e-05,45.800000,0.000,50.560,1,0,1,0.000,0.000048,45.800000
2,france,DIP,143612,0.058690,9.783000e-04,9.783000,0.000,54.812,0,0,0,0.000,0.000978,9.783000
1,france,DOC,96395,3.400000,1.000000e-01,205000.000000,0.045,262.493,257,257,0,0.003,0.100000,200.000000
4,GRQA,DOC,499007,3.349500,7.000000e-03,3700.000189,0.012,883.729,51,50,1,0.000,0.013000,870.840000
2,arcticdeltas,DOC,519,0.005760,2.100000e-03,0.023500,0.000,0.122,0,0,0,0.000,0.002100,0.023500


##### optional save informations regarding number of outliers per fraction and dataset etc.:

In [9]:
outliers_table.groupby('fraction').sum()

,dataset,n_obs,median,min,max,low_thres,up_thres,n_outliers,up_out,low_out,percent,min_without_outl,max_without_outl
fraction,,,,,,,,,,,,,
DIN,sweden,100,0.016000,0.001000,0.236000,0.000,13.676,0,0,0,0.000,0.001000,0.236000
DIP,arcticdeltasdenmarkfrancegermanyGRQAsweden,1269031,0.189245,0.001468,540.394030,0.0000.0000.0000.0000.0000.000,0.86950.56054.81221.893500.6631.384,46,2,44,0.0000.0000.0000.0000.0000.000,0.001791,477.394030
DOC,arcticdeltasdenmarkfrancegermanyGRQAsweden,870851,37.393593,0.447920,211026.290355,0.0000.1770.0450.0980.0120.958,0.122183.880262.493308.202883.729381.793,325,314,11,0.0000.0020.0030.0000.0000.001,2.637767,1680.030167
NH4N,arcticdeltasdenmarkgermanyGRQAswedenUSGS,1125475,0.303998,0.000650,90571.981046,0.0000.0000.0000.0000.0000.000,27.286127.065363.032197.5474.429264.023,13,11,2,0.0000.0000.0000.0000.0000.000,0.000663,396.317237
NO2N,germanyGRQA,936997,0.042172,0.000103,2700.000000,0.0000.000,7.15851.190,64,60,4,0.0000.000,0.000293,57.100000
NO2N_NO3N,sweden,5412,0.017250,0.000885,0.787440,0.000,1.734,0,0,0,0.000,0.000885,0.787440
NO3N,arcticdeltasdenmarkfrancegermanyGRQAUSGS,2003409,10.792363,0.004424,10738.318192,0.0000.0010.0070.0100.0000.000,147.0765734.232653.924807.47710515.32412429.300,151,0,151,0.0000.0000.0000.0000.0000.000,0.026800,10738.318192
OPO4,USGS,139759,0.036000,0.000500,6.800000,0.000,160.484,0,0,0,0.000,0.000500,6.800000
TOC,denmarkgermanyGRQAswedenUSGS,657864,41.845000,2.174000,101566.440000,0.4330.1830.0190.8740.030,160.771188.1531541.891348.134622.756,103,96,7,0.0000.0000.0000.0000.000,3.272543,2033.440000


In [10]:
outliers_table.to_csv('../../output_data/outlier_removed_raw_data/overview_identified_outliers_fraction_dataset.csv')